In [ ]:
import numpy as np
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

class CachePredictor:
    def __init__(self):
        # We use SGDRegressor for "Online Learning" (Partial Fit)
        # This allows the model to learn as the simulation runs.
        self.model = SGDRegressor(learning_rate='constant', eta0=0.01)
        self.scaler = StandardScaler()
        self.is_trained = False

    def _extract_features(self, pc, access_count, stride):
        # Feature vector: [Program Counter, Frequency, Stride]
        return np.array([[pc, access_count, stride]])

    def predict_rri(self, pc, access_count, stride):
        """Predicts the Re-Reference Interval (Lower = Use sooner)"""
        if not self.is_trained:
            return 7  # Return a default middle-ground RRI if not yet trained
        
        X = self._extract_features(pc, access_count, stride)
        X_scaled = self.scaler.transform(X)
        prediction = self.model.predict(X_scaled)
        return max(0, int(prediction[0]))

    def update_model(self, pc, access_count, stride, actual_rri):
        """Updates the model weights based on actual hardware outcome"""
        X = self._extract_features(pc, access_count, stride)
        y = np.array([actual_rri])
        
        # Partially fit the scaler and model (Online Learning)
        if not self.is_trained:
            self.scaler.fit(X)
            self.is_trained = True
        else:
            self.scaler.partial_fit(X)
            
        X_scaled = self.scaler.transform(X)
        self.model.partial_fit(X_scaled, y)

# --- Example Usage in your Simulation ---
# predictor = CachePredictor()
# prediction = predictor.predict_rri(pc=0x1234, access_count=5, stride=4)
# print(f"Predicted RRI: {prediction}")